In [1]:
import datetime
import json, re, itertools
import numpy as np
import pandas as pd
from osgeo import gdal, osr
import matplotlib.pyplot as plt
import ipyleaflet

In [2]:
import geopandas as gpd

In [3]:
import ee

In [4]:
ee.Authenticate(force=True)

Enter verification code:  4/1AdkVLPyZ-PEjxsY0j99MAPe-2aAJYyYxUuLXWO_PKWKUxu37MDPlmQRUjmw



Successfully saved authorization token.


In [6]:
ee.Initialize(project='ee-sinsteticlog')

In [7]:
import geemap
Map = geemap.Map()

In [8]:
# =========================
# UTILS
# =========================
def safe_median(ic, band_names):
    return ee.Image(
        ee.Algorithms.If(
            ic.size().gt(0),
            ic.median(),
            ee.Image.constant([0]*len(band_names))
              .rename(band_names)
              .updateMask(ee.Image(0))))

def safe_percentile(ic, band_names, p=95):

    return ee.Image(
        ee.Algorithms.If(
            ic.size().gt(1),
            ic.reduce(ee.Reducer.percentile([p])).rename(band_names),
            ee.Image.constant([0] * len(band_names))
                .rename(band_names)
                .updateMask(ee.Image(0))))

def apply_forest_mask(density):
    def mask(img):
        forest = tcd.gte(density)
        return img.updateMask(forest)
    return mask

def get_windows(t, window_days=7):
    t = ee.Date(t)
    
    monitoring_start = t
    monitoring_end = t.advance(window_days, 'day')
    timestamp = monitoring_end.advance(-1, 'day')
    
    buffer_end = monitoring_start
    buffer_start = buffer_end.advance(-2, 'month')
    
    hist_end = buffer_start
    hist_start = hist_end.advance(-2, 'year')
    
    return (monitoring_start, monitoring_end,
            buffer_start, buffer_end,
            hist_start, hist_end,
            timestamp)


# =========================
# GENERATE DATES
# =========================
def generate_dates(start, end):
    start = ee.Date(start)
    end = ee.Date(end)
    
    n_weeks = end.difference(start, 'week').floor()
    weeks = ee.List.sequence(0, n_weeks)
    
    return weeks.map(lambda w: start.advance(w, 'week'))

In [9]:
# =========================
# CLOUD MASK PROBABILITY
# =========================

CLOUD_FILTER = 60       # filtro globale CLOUDY_PIXEL_PERCENTAGE
CLD_PRB_THRESH = 50     # probabilità nuvola
NIR_DRK_THRESH = 0.15   # soglia NIR per ombre
CLD_PRJ_DIST = 1        # distanza proiezione ombra in pixel
BUFFER = 50             # dilatazione finale in metri


# Aggiunge banda cloud probability e cloud mask
def add_cloud_bands(img):
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')
    return img.addBands(ee.Image([cld_prb, is_cloud]))


# Aggiunge banda dark pixels, cloud projection e shadows
def add_shadow_bands(img):
    # Identify water pixels from the SCL band.
    not_water = img.select('SCL').neq(6)

    # Identify dark NIR pixels that are not water (potential cloud shadow pixels).
    SR_BAND_SCALE = 1e4
    dark_pixels = img.select('B8').lt(NIR_DRK_THRESH*SR_BAND_SCALE).multiply(not_water).rename('dark_pixels')

    # Determine the direction to project cloud shadow from clouds (assumes UTM projection).
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')));

    # Project shadows from clouds for the distance specified by the CLD_PRJ_DIST input.
    cld_proj = (img.select('clouds').directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST*10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 100})
        .select('distance')
        .mask()
        .rename('cloud_transform'))

    # Identify the intersection of dark pixels with cloud shadow projection.
    shadows = cld_proj.multiply(dark_pixels).rename('shadows')

    # Add dark pixels, cloud projection, and identified shadows as image bands.
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))


# Combina cloud + shadow mask finale
def add_cld_shdw_mask(img):
    # Add cloud component bands.
    img_cloud = add_cloud_bands(img)

    # Add cloud shadow component bands.
    img_cloud_shadow = add_shadow_bands(img_cloud)

    # Combine cloud and shadow mask, set cloud and shadow as value 1, else 0.
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)

    # Remove small cloud-shadow patches and dilate remaining pixels by BUFFER input.
    # 20 m scale is for speed, and assumes clouds don't require 10 m precision.
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER*2/20)
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 20})
        .rename('cloudmask'))

    # Add the final cloud-shadow mask to the image.
    return img_cloud_shadow.addBands(is_cld_shdw)


#def apply_mask(img):
#    return img.updateMask(img.select('cloudmask').Not())

    
def apply_mask(img):
    no_cloud_shadow = img.select('cloudmask').Not()
    no_snow_ice = img.select('SCL').neq(11)  # SCL class 11 = snow/ice
    return img.updateMask(no_cloud_shadow).updateMask(no_snow_ice)


In [25]:
# =========================
# S1 CHANGE
# =========================
def compute_change_s1(t, threshold, bounds_ee, orbit):

    monitoring_start, monitoring_end, buffer_start, buffer_end, hist_start, hist_end, timestamp = get_windows(t)

    s1_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(bounds_ee)
        .filterDate(hist_start, monitoring_end)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.eq('orbitProperties_pass', orbit))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select(['VV', 'VH'])
        .map(apply_forest_mask(40))
        .map(lambda img: img.clip(bounds_ee)))



    post = safe_median(s1_col.filterDate(monitoring_start, monitoring_end), ['VV','VH']).rename(['VV_post','VH_post'])

    hist_col = s1_col.filterDate(hist_start, hist_end)
    hist_median = safe_median(hist_col, ['VV','VH']).rename(['VV_hist','VH_hist'])
    hist_std = hist_col.reduce(ee.Reducer.stdDev()).rename(['VV_stdDev','VH_stdDev'])

    
    dVV_px = post.select('VV_post').subtract(hist_median.select('VV_hist')).rename('dVV')
    dVH_px = post.select('VH_post').subtract(hist_median.select('VH_hist')).rename('dVH')
    r_px = dVV_px.pow(2).add(dVH_px.pow(2)).sqrt().rename('r')
    
    delta_stack = ee.Image.cat([dVV_px, dVH_px, r_px])


    # =========================
    # SEGMENTAZIONE
    # =========================
    seeds = ee.Algorithms.Image.Segmentation.seedGrid(10)
    snic = ee.Algorithms.Image.Segmentation.SNIC(image=post.select(['VV_post','VH_post']).float(),
                                                 compactness=0.3,
                                                 connectivity=8,
                                                 neighborhoodSize=32,
                                                 seeds=seeds)

    segments= snic.select('clusters').rename('segments')


    delta_obj = delta_stack.addBands(segments).reduceConnectedComponents(reducer=ee.Reducer.mean(), labelBand='segments')

    dVV = delta_obj.select('dVV')
    dVH = delta_obj.select('dVH')
    r   = delta_obj.select('r')


    return (ee.Image.cat([dVV, dVH, r]).set('system:time_start', timestamp.millis()))

In [26]:
# =========================
# S2 INDICI
# =========================
def add_indices(img):
    ndmi = img.normalizedDifference(['B8','B11']).rename('NDMI')
    nbr  = img.normalizedDifference(['B8','B12']).rename('NBR')
    ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
    ngrdi = img.normalizedDifference(['B3','B4']).rename('NGRDI')
    return img.addBands([ndmi, nbr, ndvi, ngrdi])

# =========================
# S2 COLLECTION (con cloud mask SERIA)
# =========================
def build_s2_collection(bounds_ee, start, end):
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(bounds_ee)
          .filterDate(start, end)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60)))

    s2_clouds = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
                 .filterBounds(bounds_ee)
                 .filterDate(start, end))

    joined = ee.ImageCollection(
        ee.Join.saveFirst('s2cloudless').apply(
            primary=s2,
            secondary=s2_clouds,
            condition=ee.Filter.equals(
                leftField='system:index',
                rightField='system:index'
            )
        )
    )

    return (joined
        .map(add_cld_shdw_mask)
        .map(apply_mask)
        .map(add_indices)
        .map(apply_forest_mask(40))
        .select(['NDMI','NBR','NDVI', 'NGRDI'])
        .map(lambda img: img.clip(bounds_ee)))




def compute_change_s2_short(t, bounds_ee, weeks_back=2):

    monitoring_start, monitoring_end, _, _, _, _, timestamp = get_windows(t)

    s2 = build_s2_collection(
        bounds_ee,
        monitoring_start.advance(-weeks_back * 7, 'day'),
        monitoring_end
    )

    bands = ['NDMI', 'NBR', 'NDVI', 'NGRDI']

    # =========================
    # POST
    # =========================
    post = safe_median(
        s2.filterDate(monitoring_start, monitoring_end),
        bands
    )

    # =========================
    # HISTORICAL
    # =========================
    hist_short = s2.filterDate(
        monitoring_start.advance(-weeks_back * 7, 'day'),
        monitoring_start
    )

    hist_median = safe_median(hist_short, bands)
    hist_std = hist_short.reduce(ee.Reducer.stdDev())

    # =========================
    # DELTA PIXEL-LEVEL
    # =========================
    dNDMI_px  = post.select('NDMI').subtract(hist_median.select('NDMI')).rename('dNDMI')
    dNBR_px   = post.select('NBR').subtract(hist_median.select('NBR')).rename('dNBR')
    dNDVI_px  = post.select('NDVI').subtract(hist_median.select('NDVI')).rename('dNDVI')
    dNGRDI_px = post.select('NGRDI').subtract(hist_median.select('NGRDI')).rename('dNGRDI')

    # =========================
    # MAGNITUDES (pixel)
    # =========================
    VL_px = dNDMI_px.pow(2).add(dNBR_px.pow(2)).add(dNDVI_px.pow(2)).add(dNGRDI_px.pow(2)).sqrt().rename('VL')
    VLnir_px = dNDMI_px.pow(2).add(dNBR_px.pow(2)).sqrt().rename('VLnir')

    delta_stack = ee.Image.cat([
        dNDMI_px, dNBR_px, dNDVI_px, dNGRDI_px, VL_px, VLnir_px
    ])

    # =========================
    # (OPZIONALE) NORMALIZZAZIONE
    # =========================
    # eps = 0.01
    # delta_stack = delta_stack.divide(hist_std.add(eps))

    # =========================
    # SEGMENTAZIONE SU DELTA
    # =========================
    seeds = ee.Algorithms.Image.Segmentation.seedGrid(10)

    snic = ee.Algorithms.Image.Segmentation.SNIC(
        image=delta_stack.float(),
        compactness=0.3,
        connectivity=8,
        neighborhoodSize=32,
        seeds=seeds
    )

    segments = snic.select('clusters').rename('segments')

    # =========================
    # OBJECT-BASED (DELTA)
    # =========================
    delta_obj = delta_stack.addBands(segments).reduceConnectedComponents(
        reducer=ee.Reducer.median(),
        labelBand='segments'
    )

    dNDMI  = delta_obj.select('dNDMI')
    dNBR   = delta_obj.select('dNBR')
    dNDVI  = delta_obj.select('dNDVI')
    dNGRDI = delta_obj.select('dNGRDI')
    VL     = delta_obj.select('VL')
    VLnir  = delta_obj.select('VLnir')

    # =========================
    # OUTPUT
    # =========================
    return ee.Image.cat([
        dNDMI, dNBR, dNDVI, dNGRDI, VL, VLnir
    ]).set('system:time_start', timestamp.millis())




def compute_change_s2_seasonal(t, bounds_ee, years_back=2, week_buffer=1):

    monitoring_start, monitoring_end, _, _, _, _, timestamp = get_windows(t)

    s2 = build_s2_collection(
        bounds_ee,
        monitoring_start.advance(-years_back, 'year'),
        monitoring_end
    )

    bands = ['NDMI', 'NBR', 'NDVI', 'NGRDI']

    # =========================
    # POST
    # =========================
    post = safe_median(
        s2.filterDate(monitoring_start, monitoring_end),
        bands
    )

    # =========================
    # HISTORICAL (multi-year seasonal)
    # =========================
    def one_year(y):

        y = ee.Number(y)

        start = monitoring_start \
            .advance(y.multiply(-1), 'year') \
            .advance(-week_buffer, 'week')

        end = start.advance(7 + 2 * week_buffer, 'day')

        return safe_median(s2.filterDate(start, end), bands)

    hist_col = ee.ImageCollection(
        ee.List.sequence(1, years_back).map(one_year)
    )

    hist_median = hist_col.median()
    hist_std = hist_col.reduce(ee.Reducer.stdDev())

    # =========================
    # DELTA PIXEL-LEVEL
    # =========================
    dNDMI_px  = post.select('NDMI').subtract(hist_median.select('NDMI')).rename('dNDMI')
    dNBR_px   = post.select('NBR').subtract(hist_median.select('NBR')).rename('dNBR')
    dNDVI_px  = post.select('NDVI').subtract(hist_median.select('NDVI')).rename('dNDVI')
    dNGRDI_px = post.select('NGRDI').subtract(hist_median.select('NGRDI')).rename('dNGRDI')

    # =========================
    # MAGNITUDES
    # =========================
    VL_px = dNDMI_px.pow(2).add(dNBR_px.pow(2)).add(dNDVI_px.pow(2)).add(dNGRDI_px.pow(2)).sqrt().rename('VL')
    VLnir_px = dNDMI_px.pow(2).add(dNBR_px.pow(2)).sqrt().rename('VLnir')

    delta_stack = ee.Image.cat([
        dNDMI_px, dNBR_px, dNDVI_px, dNGRDI_px, VL_px, VLnir_px
    ])

    # =========================
    # (OPZIONALE) NORMALIZZAZIONE
    # =========================
    # eps = 0.01
    # delta_stack = delta_stack.divide(hist_std.add(eps))

    # =========================
    # SEGMENTAZIONE SU DELTA
    # =========================
    seeds = ee.Algorithms.Image.Segmentation.seedGrid(10)

    snic = ee.Algorithms.Image.Segmentation.SNIC(
        image=delta_stack.float(),
        compactness=0.3,
        connectivity=8,
        neighborhoodSize=32,
        seeds=seeds
    )

    segments = snic.select('clusters').rename('segments')

    # =========================
    # OBJECT-BASED
    # =========================
    delta_obj = delta_stack.addBands(segments).reduceConnectedComponents(
        reducer=ee.Reducer.median(),
        labelBand='segments'
    )

    dNDMI  = delta_obj.select('dNDMI')
    dNBR   = delta_obj.select('dNBR')
    dNDVI  = delta_obj.select('dNDVI')
    dNGRDI = delta_obj.select('dNGRDI')
    VL     = delta_obj.select('VL')
    VLnir  = delta_obj.select('VLnir')

    # =========================
    # OUTPUT
    # =========================
    return ee.Image.cat([
        dNDMI, dNBR, dNDVI, dNGRDI, VL, VLnir
    ]).set('system:time_start', timestamp.millis())




In [27]:
# =========================
# COSTRUZIONE COLLEZIONI
# =========================

#bounds_ee = ee.Geometry.Rectangle([27.330039, 63.905082, 27.848557, 64.117109])
bounds_ee = ee.Geometry.Rectangle([26.97262155489266533, 64.07833850843654488, 27.00644129094077428, 64.09631519162277868])

tcd = ee.Image('projects/ee-sinsteticlog/assets/TreeCoverDensity2018') #forest mask (da GEE assets)



START_DATE = '2018-08-01'
END_DATE   = '2018-10-01'

dates = generate_dates(START_DATE, END_DATE)

s1_asc = ee.ImageCollection(dates.map(lambda t: compute_change_s1(t, threshold=2.5, bounds_ee=bounds_ee, orbit='ASCENDING')))

s1_desc = ee.ImageCollection(dates.map(lambda t: compute_change_s1(t, threshold=2.5, bounds_ee=bounds_ee, orbit='DESCENDING')))

s2_short = ee.ImageCollection(dates.map(lambda t: compute_change_s2_short(t, bounds_ee, weeks_back=3)))

s2_seasonal = ee.ImageCollection(dates.map(lambda t: compute_change_s2_seasonal(t, bounds_ee, years_back=3, week_buffer=1)))

In [ ]:
==========================================================================================================================
'''DOWNLOAD TO GOOGLE DRIVE'''
==========================================================================================================================

In [14]:
def export_collection_to_drive(collection, prefix, folder, bounds_ee):

    col = collection.toList(collection.size())
    n = collection.size().getInfo()

    for i in range(n):

        img = ee.Image(col.get(i))#.toFloat()
        img = img.toFloat()
        date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()

        task = ee.batch.Export.image.toDrive(
            image=img,
            description=f'{prefix}_{date}',
            folder=folder,
            fileNamePrefix=f'{prefix}_{date}',
            region=bounds_ee,
            scale=10,
            crs='EPSG:32632',
            maxPixels=1e13,
            fileFormat='GeoTIFF')

        task.start()

In [28]:
export_collection_to_drive(s1_asc, 'R_ASC_segmented2_Area_1', 'GEE_exports', bounds_ee)

In [29]:
export_collection_to_drive(s1_desc, 'R_DESC_segmented2_Area_1', 'GEE_exports', bounds_ee)

In [30]:
export_collection_to_drive(s2_seasonal, 's2_seasonal2_segmented_Area_1', 'GEE_exports', bounds_ee)

In [31]:
export_collection_to_drive(s2_short, 's2_short_segmented2_Area_1', 'GEE_exports', bounds_ee)

In [ ]:
for task in ee.batch.Task.list()[0:7]:
    s = task.status()
    print(s['description'], s['state'])
    #task.cancel()

In [ ]:
tasks = ee.batch.Task.list()

for task in tasks:
    task.cancel()

In [ ]:
#import shapefile
#fc = ee.FeatureCollection("C:/Users/GIANDOMENICODELUCA/CNR-IBE/Dati/SINTETIC/Other_Logging_Data/Clearcuts_2019onward/Clearcuts_2019onward.shp")
gdf = gpd.read_file("C:/Users/GIANDOMENICODELUCA/CNR-IBE/Dati/SINTETIC/Other_Logging_Data/Clearcuts_2019onward/Clearcuts_2019onward_subset.shp")
gdf = gdf[['id', 'standarriv', 'geometry']]
gdf = gdf.to_crs(epsg=4326)
#Semplifica i poligoni
gdf["geometry"] = gdf["geometry"].simplify(tolerance=0.0001)  # 0.0001 gradi ≈ 10 m
fc = geemap.geopandas_to_ee(gdf)
xmin, ymin, xmax, ymax = gdf.total_bounds
bounds_ee = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])
print(xmin, ymin, xmax, ymax)


In [ ]:

def extract_timeseries(ic, fc, scale=20):

    def per_image(img):
        stats = img.reduceRegions(
            collection=fc,
            reducer=ee.Reducer.mean(),
            scale=scale
        )
        # aggiungo timestamp a ogni feature
        return stats.map(lambda f: f.set('time', img.date().format('YYYY-MM-dd')))
    
    return ic.map(per_image).flatten()


In [ ]:

ts_s2_seasonal = extract_timeseries(s2_seasonal, fc)
ts_s2_short    = extract_timeseries(s2_short, fc)
ts_s1_asc      = extract_timeseries(s1_asc, fc)
ts_s1_desc     = extract_timeseries(s1_desc, fc)


In [ ]:

task = ee.batch.Export.table.toDrive(
    collection=ts_s2_seasonal,
    description='ts_s2_seasonal',
    fileFormat='CSV')
task.start()


In [ ]:

poly_id = 461  # il tuo id

ts_one = (ts_s2_seasonal
          .filter(ee.Filter.eq('id', poly_id))
          .sort('time'))

In [ ]:
features = ts_one.getInfo()['features']

In [ ]:

df = pd.DataFrame([f['properties'] for f in features])

df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time')


In [ ]:

df['VL_smooth'] = df['VL'].rolling(25, center=True, min_periods=1).median()

plt.figure(figsize=(12,5))

plt.plot(df['time'], df['VL'], color='gray', alpha=0.4, label='VL raw')
plt.plot(df['time'], df['VL_smooth'], label='VL smooth')

plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:

bands = ['dNDMI','dNBR','dNDVI','dNGRDI']

for b in bands:
    df[b+'_smooth'] = df[b].rolling(25, center=True, min_periods=1).median()

plt.figure(figsize=(14,6))

for b in bands:
    plt.plot(df['time'], df[b+'_smooth'], label=b)

plt.legend()
plt.show()
